## Construction Rules
The NCD framework uses category theory to describe deep learning models. This raises the challenge of how we should go about encoding a mathematical system into an appropriate data structure. For instance, let's take the a mathematical construct as simple as addition. Addition is an associative operation with the structure by $+: \text{list}[\mathbb{R}] \rightarrow \mathbb{R}$. So, we create a construction rule `Addition: Numeric[] -> Numeric`. Now, whenever we have the `Addition` type, we can reference the symbols used to make it. The goal of our framework is to encode structure, and provide tools for evaluating structure as needed.

In [12]:
from abc import ABC
from dataclasses import dataclass, field
type Prod[T] = tuple[T, ...]
@dataclass(frozen=True)
class Numeric(ABC): ...
@dataclass(frozen=True)
class Addition(Numeric):
    content: Prod[Numeric]
    def __repr__(self): return ' + '.join(map(repr, self.content))

All terms are built as either (i) **construction rules** collecting other terms with a reference to some operation (eg. $+: \text{list}[\mathbb{R}] \rightarrow \mathbb{R}$), (ii) **root** terms representing specific values, or (iii) pending, **`uid`** tagged terms which may be replaced. `uid` tagged terms can be searched for in an expression, revealing degrees of freedom. By creating a bucket of `uid`s with a canonical form, we can replace and equalize terms. This allows us to encode expressions such as $f(x,x)$ by setting $y:=x$ in $f(x,y)$. This allows for high-level symbolic manipulation to be performed on terms.

This simple high-level structure already allows us to perform some algebraic manipulation, as shown in the example below.

In [13]:
import random
@dataclass(frozen=True)
class Integer(Numeric):
    value: int
    def __repr__(self): return f"'{self.value}'"
type UID = int
@dataclass(frozen=True)
class FreeNumeric(Numeric):
    uid: UID = field(default_factory=lambda: random.randint(0, 256**2))
    name: str = ''
    def __repr__(self): return self.name
def degrees_of_freedom(target: Numeric) -> Prod[FreeNumeric]:
    match target:
        case FreeNumeric(uid=uid):
            return (target,)
        case Addition(content=content):
            return tuple(n for ns in content for n in degrees_of_freedom(ns))
        case _:
            return ()
def assign(target: Numeric, bucket: Prod[UID], canonical: Numeric) -> Numeric:
    match target:
        case FreeNumeric(uid=uid) if uid in bucket:
            return canonical
        case Addition(content=content):
            return Addition(tuple(assign(n, bucket, canonical) for n in content))
        case _:
            return target
        
x = FreeNumeric(name='x')
y = FreeNumeric(name='y')
expr = Addition((x, Integer(1), y))

print('Expression:', expr)
print('Degrees of Freedom:', degrees_of_freedom(expr))
print('Assignment:', assign(expr, (x.uid, y.uid), x))

Expression: x + '1' + y
Degrees of Freedom: (x, y)
Assignment: x + '1' + x


Terms aim to provide high-level, structural capabilities which are used across frameworks. If we now wish to provide some evaluation ability on top of our construction rules, we do that separately. For instance, we can provide functionality to evaluate a `Numeric` term, or to convert it to `Sympy`. By separating concerns, we are able to provide modular code which can be easily exported across platforms and understood as mathematical expressions.

In [14]:
def evaluate(target: Numeric) -> int:
    match target:
        case Integer(value=value):
            return value
        case Addition(content=content):
            return sum(evaluate(n) for n in content)
        case _:
            raise ValueError(f"Cannot evaluate {target}")

import sympy
def to_sympy(target: Numeric) -> sympy.Expr:
    match target:
        case Integer(value=value):
            return sympy.Integer(value)
        case FreeNumeric(name=name):
            return sympy.Symbol(name)
        case Addition(content=content):
            return sum(to_sympy(n) for n in content)
        case _:
            raise ValueError(f"Cannot convert {target} to sympy")
        
print('Assigned to Integers:', integer_form := assign(expr, (x.uid, y.uid), Integer(2)))
print('Evaluated:', evaluate(integer_form))
print('Sympy Expression:', sympy_expr := to_sympy(expr))

Assigned to Integers: '2' + '1' + '2'
Evaluated: 5
Sympy Expression: x + y + 1


This package is built on term construction rules of this form. Data structures, include the foundational `Term.py` module, can be found in `../data_structures`. Here, you can also find a more fully developed `Numeric.py` class. As we add more mathematical definitions, this same term construction pattern will be followed, giving us access to general purpose algebraic assignment and allowing code to closely match mathematics.